# K-Space Fingerprinting

Every scan has a characteristic frequency distribution — a fingerprint. This notebook extracts that fingerprint **without reconstructing an image**.

Three signals are computed directly from raw k-space:

1. **Radial power profile** — how energy is distributed from low to high frequency
2. **Energy ratio** — fraction of total signal in the low-frequency center
3. **Asymmetry score** — left-right imbalance in k-space power (healthy brains are symmetric)

These are features you could feed directly into a classifier — no image, no segmentation.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from kode.io import load_kspace, list_slices
from kode.fingerprint import radial_profile, energy_ratio, asymmetry_score

In [ ]:
H5_FILE = '../data/your_file.h5'

n_slices = list_slices(H5_FILE)
print(f'File has {n_slices} slices')

kspace = load_kspace(H5_FILE, slice_idx=15)
print(f'K-space shape: {kspace.shape}')

In [ ]:
# Radial power profile
profile = radial_profile(kspace)

plt.figure(figsize=(10, 4))
plt.plot(profile, color='steelblue', linewidth=1.5)
plt.xlabel('Radial distance from k-space center (pixels)')
plt.ylabel('Mean power')
plt.title('K-Space Radial Power Profile — Fingerprint Curve')
plt.yscale('log')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../results/fingerprint_profile.png', dpi=150)
plt.show()

In [ ]:
# Compute fingerprint metrics across all slices
energy_ratios = []
asymmetry_scores = []

for i in range(n_slices):
    ks = load_kspace(H5_FILE, slice_idx=i)
    energy_ratios.append(energy_ratio(ks))
    asymmetry_scores.append(asymmetry_score(ks))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(energy_ratios, color='steelblue')
axes[0].set_title('Low-Freq Energy Ratio Across Slices')
axes[0].set_xlabel('Slice index')
axes[0].set_ylabel('Energy ratio')
axes[0].grid(alpha=0.3)

axes[1].plot(asymmetry_scores, color='tomato')
axes[1].set_title('K-Space Asymmetry Score Across Slices')
axes[1].set_xlabel('Slice index')
axes[1].set_ylabel('Asymmetry score')
axes[1].grid(alpha=0.3)

plt.suptitle('K-Space Fingerprint — Per-Slice Metrics', fontsize=13)
plt.tight_layout()
plt.savefig('../results/fingerprint_metrics.png', dpi=150)
plt.show()

print(f'Mean energy ratio:    {np.mean(energy_ratios):.4f}')
print(f'Mean asymmetry score: {np.mean(asymmetry_scores):.4f}')